In [ ]:
# 1. Install dependencies
!pip install -q transformers torch torchvision pillow pydantic matplotlib groq


In [ ]:
# 2. Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from typing import List
from pydantic import BaseModel
from transformers import ViTImageProcessor, ViTForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
# 3. Load a pretrained, open-weight plant-disease ViT
MODEL_NAME = "wambugu71/crop_leaf_diseases_vit"

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
model = ViTForImageClassification.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()
model.to(device)

print("Loaded model with", model.config.num_labels, "classes")
print("Sample labels:", list(model.config.id2label.values())[:10])


In [ ]:
# 4. Strict output schema for inter-agent communication
class VisionNodeOutput(BaseModel):
    disease_class: str
    confidence_score: float
    attention_matrix: List[List[float]]
    is_ambiguous: bool

CONFIDENCE_THRESHOLD = 0.75


In [ ]:
# 5. Attention rollout
def attention_rollout(attentions):
    tokens = attentions[0].size(-1)
    result = torch.eye(tokens, device=attentions[0].device).unsqueeze(0)
    for attn in attentions:
        attn_avg = attn.mean(dim=1)
        attn_avg = attn_avg + torch.eye(tokens, device=attn_avg.device)
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)
        result = torch.matmul(attn_avg, result)
    return result


In [ ]:
# 6. Main inference function
def run_vision_node(image: Image.Image) -> VisionNodeOutput:
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_idx = int(torch.argmax(probs))
    confidence = float(probs[pred_idx])
    label = model.config.id2label[pred_idx]

    rolled = attention_rollout(outputs.attentions)
    cls_to_patches = rolled[0, 0, 1:]
    grid_size = int(len(cls_to_patches) ** 0.5)
    attn_grid = cls_to_patches.reshape(grid_size, grid_size).cpu().numpy()
    attn_grid = (attn_grid - attn_grid.min()) / (attn_grid.max() - attn_grid.min() + 1e-8)

    return VisionNodeOutput(
        disease_class=label,
        confidence_score=round(confidence, 4),
        attention_matrix=attn_grid.tolist(),
        is_ambiguous=confidence < CONFIDENCE_THRESHOLD,
    )


In [ ]:
# 7. Heatmap visualization
def visualize_attention(image: Image.Image, result: VisionNodeOutput, save_path=None):
    attn = np.array(result.attention_matrix)
    attn_resized = np.array(
        Image.fromarray((attn * 255).astype(np.uint8)).resize(image.size, Image.BICUBIC)
    ) / 255.0

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image)
    axes[0].set_title("Original Leaf")
    axes[0].axis("off")

    axes[1].imshow(attn_resized, cmap="jet")
    axes[1].set_title("Attention Rollout")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(attn_resized, cmap="jet", alpha=0.5)
    axes[2].set_title("Overlay Map")
    axes[2].axis("off")
    if save_path:
        plt.savefig(save_path)
    plt.show()


In [ ]:
# 8. Groq Integration for Diagnostic Recommendation System
import os
from groq import Groq

# Handles API key resolution automatically across environment states
try:
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
except Exception:
    groq_api_key = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

def generate_treatment_report(vision_output: VisionNodeOutput) -> str:
    """Utilizes Groq API to convert raw ViT diagnostics into structural agronomist action items."""
    prompt = (
        f"You are a highly qualified expert AI Agronomist Advisor. Parse this diagnostic vision packet:\n"
        f"- Classification: {vision_output.disease_class}\n"
        f"- Score Certainty: {vision_output.confidence_score * 100:.2f}%\n"
        f"- Flag for Validation (Ambiguous): {vision_output.is_ambiguous}\n\n"
        f"Provide a fast, highly targeted treatment itinerary including organic mitigations and structural steps."
    )
    
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a professional agricultural consultant providing definitive plant pathobiology analysis."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.25,
            max_tokens=450
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Groq Connection Refused/Failed: {str(e)}"
